# Aula 01 — IA aplicada + Hello Modelo (Iris)


**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  


## 🎯 Objetivos

1) Entender o ambiente do Google Colab CPU/GPU  
2) Manipular dados com Pandas utilizando dataset - Iris  
3) Treinar seu primeiro modelo de ML (baseline)  
4) Entender o fluxo sagrado: **Dados → Treino → Teste**  
5) Aprender o **pipeline reprodutível** (pré-processamento + modelo)
6) Exercícios

## Teoria


- IA aplicada = resolver problema real com dados + métricas.
- Modelo = função parametrizada que generaliza padrões.
- Treino ajusta parâmetros para reduzir erros.
- Teste mede generalização em dados não vistos.
- Pipeline evita bagunça e aumenta a reprodutibilidade.


## ⚙️ Como alterar o tipo de ambiente de execução (CPU/GPU/TPU) no Google Colab

- Colab pode iniciar em CPU.
- Para trocar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
- Trocar CPU↔GPU pode reiniciar o runtime (você perde variáveis e roda tudo de novo).
- Nesta aula (Scikit-Learn), GPU **não é obrigatória** — mas vamos checar.

## Configuração do ambiente (CPU vs GPU)


- No Colab, às vezes você está em **CPU** por padrão.
- Se estiver em GPU, o comando `nvidia-smi` funciona.
- Se não estiver, ele não existe e dá erro — isso é normal.


In [1]:
#Checar GPU de forma “à prova de erro”
import shutil, subprocess, textwrap


def try_nvidia_smi():
    if shutil.which("nvidia-smi") is None:
        print("GPU NVIDIA não detectada (runtime provavelmente em CPU).")
        print("No Colab: Runtime/Ambiente de execução → Alterar tipo de hardware → GPU.")
        return
    out = subprocess.check_output(["nvidia-smi"], text=True)
    print(out)


try_nvidia_smi()


Mon Mar  9 23:35:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bibliotecas que vamos usar


- **Pandas**: “Excel do Python”
- **Seaborn**: datasets prontos + gráficos
- **Scikit-Learn**: ML clássico (split, modelos, métricas, pipelines)

In [2]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

## Dataset Iris (o “Hello World” da IA)

Aqui o objetivo é **entender o fluxo**:
- carregar dados
- olhar a tabela
- separar X e y
- dividir treino/teste
- treinar
- medir

### Existem quase 300 espécies diferentes de Iris já descobertas. Para o nosso objetivo nesta aula, vamos utilizar 3 espécies:

- Setosa
- Versicolor
- Virginica

![](https://content.codecademy.com/programs/machine-learning/k-means/iris.svg)

![](https://miro.medium.com/max/3500/1*f6KbPXwksAliMIsibFyGJw.png)


In [3]:
df = sns.load_dataset("iris")
display(df.head())
print("shape:", df.shape)
display(df["species"].value_counts())

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


shape: (150, 5)


,count
species,
setosa,50
versicolor,50
virginica,50


## 🎯 Problema (Iris)
- **X (features):** medidas da flor (sépalas/pétalas)
- **y (target):** espécie (`setosa`, `versicolor`, `virginica`)

In [4]:
# Separar X e y
X = df.drop("species", axis=1)
y = df["species"]

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

Formato de X: (150, 4)
Formato de y: (150,)


In [5]:
# Estatística básica + contagem por classe
display(df.describe())
display(df["species"].value_counts())

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


,count
species,
setosa,50
versicolor,50
virginica,50


X = df.drop("species", axis=1)

- df.drop(): Este é um método do Pandas usado para remover linhas ou colunas de um DataFrame.

- "species": Aqui, você está especificando o nome da coluna que deseja remover. No contexto do dataset Iris, "species" é a coluna que contém a espécie da flor, que é a variável que queremos prever (o "alvo" ou "target").

- axis=1: Este argumento é crucial. Ele indica que a operação de drop deve ser aplicada às colunas do DataFrame.

- Se fosse axis=0, a operação seria aplicada às linhas. Ao usar axis=1, estamos dizendo ao Pandas para "remover a coluna 'species'".

- Resultado: A variável X (que representa as features ou características) receberá um novo DataFrame contendo todas as colunas originais do df exceto a coluna "species".

## ✅ Checkpoint (EDA rápido)
Olhe:
- quantidade de linhas/colunas (`shape`)
- distribuição das classes (`value_counts`)
- se tem valores ausentes (vamos checar no Exercício A)

### Treino vs Teste (conceito crítico)

- Treino X: onde o modelo aprende padrões.
- Teste y: onde avaliamos se ele generaliza.
- Se eu avaliar no treino, posso “me enganar” (overfitting).

In [6]:
X

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [7]:
y

,species
0,setosa
1,setosa
2,setosa
3,setosa
4,setosa
...,...
145,virginica
146,virginica
147,virginica
148,virginica


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
acc = accuracy_score(y_test, pred)

print(f"Acurácia (Iris, split único): {acc:.4f}")

Acurácia (Iris, split único): 1.0000


Em resumo, a DecisionTreeClassifier é um algoritmo poderoso para encontrar limites de decisão claros, e o dataset Iris possui características que permitem que esses limites sejam encontrados com alta precisão, resultando em uma classificação perfeita para o conjunto de teste específico gerado.

É importante notar que 100% de acurácia é raro em problemas do mundo real e, em contextos mais complexos, poderia ser um sinal de overfitting (o modelo "decorou" os dados de treino e não generaliza bem), mas no Iris, é mais um reflexo da simplicidade e separabilidade dos dados

In [9]:
print(f"Acurácia (Iris): {acc*100:.2f}%")

Acurácia (Iris): 100.00%


Natureza do Dataset Iris: O dataset Iris é um conjunto de dados clássico e relativamente "fácil" para muitos algoritmos de classificação.

As classes de flores (setosa, versicolor, virginica) são, em grande parte, linearmente separáveis ou separáveis por regras simples baseadas nas características das pétalas e sépalas.

A espécie "setosa", por exemplo, é quase sempre perfeitamente distinguível das outras.

## ⚠️ Importante: 100% de acurácia nem sempre é “bom”
- Pode ser split “sortudo”
- Pode ser overfitting (modelo decorou padrões do treino)
- Pode ser vazamento de dados (leakage) em problemas reais
✅ Vamos confirmar com validação cruzada na sequência.

### Teste “mundo real”

In [10]:
nova_flor = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=X.columns)
print("Previsão para nova flor:", modelo.predict(nova_flor)[0])

Previsão para nova flor: setosa


A **DecisionTreeClassifier** (Árvore de Decisão para Classificação) é um algoritmo de Machine Learning supervisionado usado para resolver problemas de classificação.

Como funciona:

- Ela constrói um modelo em forma de árvore, onde cada nó interno representa um teste em uma característica (feature).
- Cada ramo representa o resultado desse teste.
- Cada nó folha (terminal) representa uma decisão de classe.

Características principais:

- Intuitiva e interpretável: A lógica de decisão pode ser visualizada e entendida facilmente.
- Versátil: Lida bem com dados numéricos e categóricos.
- Base para outros modelos: É o componente fundamental de algoritmos mais avançados, como Random Forests e Gradient Boosting.
- Em essência, ela "aprende" uma série de regras "se-então" a partir dos dados para prever a classe de novas observações.

# 🧪 Exercício A — Entendendo os dados (Iris)

## Faça:
1) Mostre:
- `df.shape`
- `df.info()`
- `df.isna().sum()`

2) Mostre:
- `df["species"].value_counts()`

3) Escreva **2 linhas** explicando:
- **o que é X (features)** e **o que é y (target)** nesse dataset Iris

## Evidência mínima (saída):
- prints/outputs do `shape`, `isna().sum()` e `value_counts()`

In [21]:
# 1. Visualizando informações do dataframe
print(f"shape: {df.shape}\n")
print(f"info:")
print(df.info(), "\n")
print("sum:")
print(df.isna().sum())

shape: (150, 5)

info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB
None 

sum:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


In [22]:
# 2. Contagem de espécies
df['species'].value_counts()

,count
species,
setosa,50
versicolor,50
virginica,50


3. Eixos X e Y:</br>
X representa os dados que serão usados para identificar os padrões do modelo, e o Y representa o Label daquele conjunto de padrões. Nesse exemplo, X representa as medidas das pétalas das flores, e Y representa a espécie que corresponde aquelas medidas.

# 🧪 Exercício B — Seu primeiro baseline (Iris)

## Objetivo
Comparar como a **divisão treino/teste** impacta a acurácia do modelo.

## Faça (duas comparações)

### 1) Comparar `test_size` (mantendo `random_state=42`)
Rode o mesmo modelo duas vezes:
- `test_size=0.2`
- `test_size=0.3`

✅ **Você deve obter 2 acurácias** (uma para cada `test_size`).

### 2) Comparar `random_state` (mantendo `test_size=0.2`)
Rode o mesmo modelo duas vezes:
- `random_state=42`
- `random_state=7`

✅ **Você deve obter 2 acurácias** (uma para cada `random_state`).

## Regras
- Use o mesmo modelo: `DecisionTreeClassifier(random_state=42)`
- Use o dataset `iris`
- Não altere as features nem o target (X e y)

## Evidência mínima (saída)
- **2 acurácias (test_size diferente)** + **2 acurácias (random_state diferente)**

In [47]:
# 1. Comparar test_size

# test_size 0.2
X_train_02, X_test_02, y_train_02, y_test_02 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo_02 = DecisionTreeClassifier(random_state=42)
modelo_02.fit(X_train_02, y_train_02)

pred_02 = modelo_02.predict(X_test_02)
acc_02 = accuracy_score(y_test_02, pred_02)

print(f"1. Acurácia (Iris, split único): {acc_02:.4f}")

# test_size 0.3
X_train_03, X_test_03, y_train_03, y_test_03 = train_test_split(
    X, y, test_size=0.3, random_state=42
)

modelo_03 = DecisionTreeClassifier(random_state=42)
modelo_03.fit(X_train_03, y_train_03)

pred_03 = modelo_03.predict(X_test_03)
acc_03 = accuracy_score(y_test_03, pred_03)

print(f"2. Acurácia (Iris, split único): {acc_03:.4f}")

print("\nComo o modelo é pequeno, não existe diferença entre 0.2 e 0.3.")
print("Alterando 'test_size' para >= 0.4 é possível verificar a diferença.")

1. Acurácia (Iris, split único): 1.0000
2. Acurácia (Iris, split único): 1.0000

Como o modelo é pequeno, não existe diferença entre 0.2 e 0.3.
Alterando 'test_size' para >= 0.4 é possível verificar a diferença.


In [51]:
# 2. Comparar random_state
X_train_07, X_test_07, y_train_07, y_test_07 = train_test_split(
    X, y, test_size=0.2, random_state=7
)

modelo_07 = DecisionTreeClassifier(random_state=42)
modelo_07.fit(X_train_07, y_train_07)

pred_07 = modelo_07.predict(X_test_07)
acc_07 = accuracy_score(y_test_07, pred_07)

print(f"1. Acurácia (Iris, split único): {acc:.4f}")
print(f"2. Acurácia (Iris, split único): {acc_07:.4f}")

1. Acurácia (Iris, split único): 1.0000
2. Acurácia (Iris, split único): 0.9000
